# ASDEA_sensors -- MASTER tour (runnable)

Runs end to end against the example folder. It shows the whole API and links the
10 focused notebooks, whose examples are represented here too.
Everything uses a short window so it runs fast.

## Focused notebooks

| # | Notebook | Topic |
|---|----------|-------|
| 01 | load_and_inspect | load, geometry, summary, prints |
| 02 | windows_and_export | time windows + export to .h5 |
| 03 | signal_pipeline | baseline / filter / derive |
| 04 | filtered_vs_unfiltered | raw vs filtered spectra |
| 05 | response_spectra | Newmark, RotD |
| 06 | intensity_measures | Arias, CAV, Housner, peaks |
| 07 | frequency_content | Fourier, PSD, STFT |
| 08 | building_modal | transfer function, coherence, mode shapes |
| 09 | torsion_and_drift | torsion, drift, base rocking |
| 10 | ambient_and_amplification | ambient step-by-step, HVSR, amplification |

## Setup, geometry, dataset

In [1]:
import sys
sys.path.insert(0, "../src")            # run from the examples/ folder

import numpy as np
from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

# The example data folder (edit to your path if needed).
DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"

# A short window keeps every example fast (one ~10 min file).
WSTART, WLEN = "2026-05-31 15:00:00", "10min"

In [2]:
# Sensor geometry up front (approximate UTM E/N, elevation). Building methods
# (torsion, mode shapes, drift) need these references.
settings.SENSOR_GEOMETRY["MOF00134"].update(E=500000.0, N=6300000.0, elev=0.0)
settings.SENSOR_GEOMETRY["MNAT0031"].update(E=500000.0, N=6300000.0, elev=7.0)
settings.SENSOR_GEOMETRY["MNAT0034"].update(E=500000.0, N=6300000.0, elev=10.5)
settings.SENSOR_GEOMETRY["MOF00135"].update(E=500000.0, N=6300000.0, elev=14.0)
settings.SENSOR_GEOMETRY["MOF00136"].update(E=500006.0, N=6300004.0, elev=14.0)
{d: settings.SENSOR_GEOMETRY[d]["floor"] for d in settings.SENSOR_GEOMETRY}

{'MOF00134': -1, 'MNAT0031': 2, 'MNAT0034': 3, 'MOF00135': 4, 'MOF00136': 4}

In [3]:
ds = SensorDataset(DATA, verbose=True)
ds.devices

------------------------------------------------------------
SensorDataset
------------------------------------------------------------
path        : C:\Users\ppala\Desktop\02_31MAY2026
files       : 32
time span   : 2026-05-31 14:52:12  ->  2026-05-31 20:02:13
duration    : 18601.0 s
devices     : MNAT0031, MNAT0034, MOF00134, MOF00135, MOF00136
fs / dt     : 252.5885 Hz / 0.003959 s
------------------------------------------------------------
axes (per sensor):
  MNAT0031   -> (3, 1, 5)
  MNAT0034   -> (3, 1, 5)
  MOF00134   -> (0, 1, 2)
  MOF00135   -> (3, 1, 5)
  MOF00136   -> (3, 1, 5)
------------------------------------------------------------
on-disk size: 782.17 MB
RAM         : used 33.12 GB / avail 0.40 GB (99%)
------------------------------------------------------------


['MNAT0031', 'MNAT0034', 'MOF00134', 'MOF00135', 'MOF00136']

## Window + decoupled pipeline

In [4]:
w = ds.MOF00135.window(WSTART, WLEN)
sig = w.signal(components="all").baseline().filter(0.1, 24.9).derive()
print("n =", sig.n, " dt =", round(sig.dt, 6), " vel filled:", sig.vel_x is not None)

C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


[signal] MOF00135 n=149240 dt=0.004021 comps=all


n = 149240  dt = 0.004021  vel filled: True


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")


## Seismic (filtered window)

In [5]:
wf = w.filter(0.1, 24.9)
spec = wf.newmark(component="x")
print("PSa max =", round(float(spec["PSa"].max()), 5), "m/s^2,", len(spec["T"]), "periods")
print("arias:", {k: (round(float(v),3) if np.isscalar(v) else "...") for k, v in wf.arias(component="x").items()})
print("CAV =", round(float(wf.cav(component="x")["CAV"]), 5))
print("peaks x =", {k: round(float(v),5) for k, v in wf.peaks(component="all")["x"].items()})
print("dominant freqs =", [round(float(x),2) for x in wf.fourier(component="x")["dom_freqs"]])

[newmark] MOF00135 comp=x zeta=0.05 Tmax=5.01 dT=0.01 -> 501 periods (computed)
PSa max = 0.00938 m/s^2, 501 periods
[arias] MOF00135 comp=x low=5 high=95 (computed)
arias: {'IA_percent': '...', 't_start': 31.012, 't_end': 592.695, 'IA_total': 0.0, 'pot_dest': 0.0}
[cav] MOF00135 comp=x (computed)
CAV = 0.29345
[peaks] MOF00135 comp=x PGA=0.005078 (computed)
[peaks] MOF00135 comp=y PGA=0.00257 (computed)
[peaks] MOF00135 comp=z PGA=0.008002 (computed)
peaks x = {'PGA': 0.00508, 'PGV': 0.00167, 'PGD': 0.02357}
[fourier] MOF00135 comp=x nfreq=4 smooth=None (computed)
dominant freqs = [0.2, 14.51, 9.46, 17.77]


## Building (windowed signals)

In [6]:
from asdea_sensors.building import transfer_function, coherence
top  = ds.MOF00135.window(WSTART, WLEN).filter(0.1, 24.9).signal(components="x")
base = ds.MOF00134.window(WSTART, WLEN).filter(0.1, 24.9).signal(components="x")
frf = transfer_function.compute(top.acc_x, base.acc_x, top.dt, smooth="konno")
print("modal freqs (Hz):", [round(float(x),2) for x in frf["modal_freqs"][:5]])
coh = coherence.compute(top.acc_x, base.acc_x, top.dt)
print("coherence points:", len(coh["f"]))

[signal] MOF00135 n=149240 dt=0.004021 comps=x


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(
C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")


[signal] MOF00134 n=149340 dt=0.004018 comps=x


modal freqs (Hz): [0.24, 0.97, 1.46, 2.43, 3.16]
coherence points: 513


## Ambient (step by step)

In [7]:
config = {"Fs": ds.fs, "STA": 1.0, "LTA": 30.0,
          "vent": 20.0, "vmin": 0.0, "vmax": 1000.0,   # wide -> keep all windows
          "p": 0.05, "bexp": 40, "f1": 0.1, "f2": 25.0, "vent_seismic": False}
config

{'Fs': 252.58854665265045,
 'STA': 1.0,
 'LTA': 30.0,
 'vent': 20.0,
 'vmin': 0.0,
 'vmax': 1000.0,
 'p': 0.05,
 'bexp': 40,
 'f1': 0.1,
 'f2': 25.0,
 'vent_seismic': False}

In [8]:
amb = ds.MOF00135.window(WSTART, WLEN).filter(0.1, 24.9).signal().ambient(config, component="x")
amb.average()
print("mean spectrum points:", amb.mean_spectrum.shape[0], " dominant period:", round(float(amb.dominant_period),3), "s")

[signal] MOF00135 n=149240 dt=0.004021 comps=all
- sta_lta: ratio computed (STA=1 s, LTA=30 s)
- select_windows: 28 window(s) selected
- taper: Tukey taper applied (p=0.05)
- fft: per-window FFT computed


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(
C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")


- smooth: Konno-Ohmachi smoothing applied (bexp=40)
- average: mean spectrum computed (T=1.2498 s)
mean spectrum points: 2525  dominant period: 1.25 s


## Export to .h5 with Provenance

In [9]:
import tempfile, os
out = os.path.join(tempfile.gettempdir(), "asdea_master.h5")
wf.export_h5(out)
from asdea_sensors.io.results_file import ResultsFile
r = ResultsFile(out)
print("devices:", r.devices, " units:", r.provenance.get("units"))

devices: ['MOF00135']  units: SI


## A plot

In [10]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,3))
plt.plot(spec["T"], spec["PSa"]); plt.xlabel("T [s]"); plt.ylabel("PSa [m/s^2]")
plt.title("Newmark PSa - MOF00135 x"); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

C:\Users\ppala\AppData\Local\Temp\claude\ipykernel_839328\3913790437.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Newmark PSa - MOF00135 x"); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
